In [1]:
from pathlib import Path
import zipfile
import urllib.request
import shutil
import pandas as pd


def download_and_prepare_far_trans(root: Path) -> None:
    """
    Download, extract, and prepare the FAR-Trans dataset.

    The function:
    - Downloads the dataset if not present
    - Extracts contents into the root directory
    - Flattens nested FAR-Trans folder if present
    - Overwrites existing files safely
    - Removes the zip file after extraction
    """
    root.mkdir(parents=True, exist_ok=True)

    zip_url = "https://researchdata.gla.ac.uk/1658/1/FAR-Trans.zip"
    zip_path = root / "FAR-Trans.zip"

    if not zip_path.exists():
        urllib.request.urlretrieve(zip_url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(root)

    nested_folder = root / "FAR-Trans"

    if nested_folder.exists() and nested_folder.is_dir():
        for item in nested_folder.iterdir():
            destination = root / item.name

            if destination.exists():
                if destination.is_file():
                    destination.unlink()
                else:
                    shutil.rmtree(destination)

            shutil.move(str(item), root)

        shutil.rmtree(nested_folder)

    zip_path.unlink(missing_ok=True)


def find_file(root: Path, filename: str) -> Path:
    """
    Locate a file inside the dataset root directory.

    Args:
        root: Root dataset directory.
        filename: Name of the file to locate.

    Returns:
        Path to the located file.

    Raises:
        FileNotFoundError: If file cannot be found.
    """
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(f"Couldn't find {filename} under {root}")

    if len(matches) > 1:
        print(f"Warning: multiple matches for {filename}, using {matches[0]}")

    return matches[0]


def load_far_trans_dataset(root: Path) -> dict[str, pd.DataFrame | str]:
    """
    Load all FAR-Trans dataset files into pandas objects.

    Args:
        root: Root dataset directory.

    Returns:
        Dictionary containing loaded DataFrames and questionnaire text.
    """
    files = [
        "transactions.csv",
        "customer_information.csv",
        "asset_information.csv",
        "markets.csv",
        "close_prices.csv",
        "limit_prices.csv",
        "questionnaires.csv",
    ]

    paths = {fn: find_file(root, fn) for fn in files}

    data = {
        "transactions": pd.read_csv(
            paths["transactions.csv"], parse_dates=["timestamp"]
        ),
        "customers": pd.read_csv(
            paths["customer_information.csv"],
            parse_dates=["lastQuestionnaireDate", "timestamp"],
        ),
        "assets": pd.read_csv(
            paths["asset_information.csv"], parse_dates=["timestamp"]
        ),
        "markets": pd.read_csv(paths["markets.csv"]),
        "close_prices": pd.read_csv(
            paths["close_prices.csv"], parse_dates=["timestamp"]
        ),
        "limit_prices": pd.read_csv(
            paths["limit_prices.csv"], parse_dates=["minDate", "maxDate"]
        ),
        "questionnaire": paths["questionnaires.csv"].read_text(
            encoding="utf-8", errors="replace"
        ),
    }

    return data

In [2]:
ROOT = Path("../data")

download_and_prepare_far_trans(ROOT)

dataset = load_far_trans_dataset(ROOT)

transactions = dataset["transactions"]
customers = dataset["customers"]
assets = dataset["assets"]
markets = dataset["markets"]
limit_prices = dataset["limit_prices"]
questionnaire = dataset["questionnaire"]

In [4]:
customers.head(3)

,customerID,customerType,riskLevel,investmentCapacity,lastQuestionnaireDate,timestamp
0,DED5BF19E23CCCFEE322,Premium,Balanced,CAP_80K_300K,2021-11-30,2021-03-19
1,DED5BF19E23CCCFEE322,Premium,Balanced,CAP_80K_300K,2021-11-30,2022-01-21
2,6C0C752E66D5F0486C71,Mass,Income,Predicted_CAP_LT30K,2015-04-27,2018-01-02


In [5]:
transactions.head(3)

,customerID,ISIN,transactionID,transactionType,timestamp,totalValue,units,channel,marketID
0,00017496858921195E5A,GRS434003000,7590224,Buy,2020-03-27,11000.0,5000.0,Internet Banking,XATH
1,00017496858921195E5A,GRS434003000,7607029,Sell,2020-04-06,12080.0,5000.0,Internet Banking,XATH
2,00017496858921195E5A,GRS434003000,7634872,Buy,2020-04-24,13400.0,5000.0,Internet Banking,XATH


In [6]:
assets.head(3)

,ISIN,assetName,assetShortName,assetCategory,assetSubCategory,marketID,sector,industry,timestamp
0,GRF000011004,DHLOS PET OTE,DELPOIB,MTF,Balanced,AEDAK,NaN,NaN,2018-01-02
1,GRF000014008,DELOS FIXED INCOME PLUS - BOND FUND,DELDBDF,MTF,Structured,AEDAK,NaN,NaN,2018-01-02
2,GRF000022001,ΔΗΛΟΣ ΒΡΑΧΥΠΡΟΘΕΣΜΩΝ & ΜΕΣΟΠΡΟΘΕΣΜΩΝ ΕΠΕΝΔΥΣΕΩ...,DELPIMM,MTF,Money Market,AEDAK,NaN,NaN,2018-01-02


In [7]:
markets.head(3)

,exchangeID,marketID,name,description,country,tradingDays,tradingHours,marketClass
0,AEDAK,AEDAK,AEDAK Funds,Mutual funds offered by EFG Eurobank and Deuts...,Greece,"Mon,Tue,Wed,Thu,Fri",08:15-15:20,Public Securities
1,XAMS,ALXA,Euronext - Growth Amsterdam,The Euronext is a pan-European bourse providin...,Netherlands,"Mon,Tue,Wed,Thu,Fri",08:00-16:30,Public Securities
2,XBRU,ALXB,Euronext - Growth Brussels,The Euronext is a pan-European bourse providin...,Belgium,"Mon,Tue,Wed,Thu,Fri",08:00-16:30,Public Securities


In [8]:
limit_prices.head(3)

,ISIN,minDate,maxDate,priceMinDate,priceMaxDate,profitability
0,100974271034,2018-01-02,2022-11-29,3.330,4.165,0.250751
1,BE0974271034,2018-01-02,2022-11-29,3.345,4.140,0.237668
2,BE0974293251,2018-01-02,2022-11-29,93.260,56.200,-0.397384
